# MoCo v2 (ResNet-50) Pretraining — Jupyter 서버 (RTX A5000 24GB)

baseline 비교용. 사전 준비·권장 실행 방식은 `server_train_mocov3.ipynb` 상단과 동일.
Terminal 실행:
```bash
nohup python -u scripts/train_mocov2.py > logs/train_mocov2.out 2>&1 &
```

In [ ]:
# Cell 1 — 작업 디렉토리 = 레포 루트 + GPU 확인
# (서버에는 Drive 마운트/심링크 불필요 — data/outputs/logs/features는 레포 루트에 영구 보존)
import os, torch
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
os.chdir(ROOT)
print('Working dir:', os.getcwd())

assert torch.cuda.is_available(), 'GPU 사용 불가 — CUDA 환경(드라이버/torch cuda wheel)을 확인하세요.'
print(f'GPU : {torch.cuda.get_device_name(0)}')   # 기대값: NVIDIA RTX A5000
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'BF16: {torch.cuda.is_bf16_supported()}')  # A5000(Ampere) → True 필수

In [ ]:
# Cell 2 — 학습 시작 (자동 resume + 실시간 로그 출력)
import glob, re, subprocess

OUT_DIR = 'outputs/mocov2_r50_seed42'

def ep_num(p):
    return int(re.search(r'ckpt_ep(\d+)\.pth', p).group(1))
ckpts = sorted(glob.glob(f'{OUT_DIR}/ckpt_ep*.pth'), key=ep_num)

if ckpts:
    print(f'Resume: {ckpts[-1]}')
    resume_args = ['--resume', ckpts[-1]]
else:
    print('처음부터 학습 시작')
    resume_args = []

cmd = ['python3', '-u', 'scripts/train_mocov2.py'] + resume_args

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
print(f'\n학습 종료 (exit code: {proc.returncode})')

In [ ]:
# Cell 3 — 학습 상태 확인
!echo '=== 최근 로그 ==='
!tail -n 20 logs/mocov2_seed42.log
!echo ''
!echo '=== 저장된 체크포인트 ==='
!ls -lh outputs/mocov2_r50_seed42/